In [ ]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import pytz
import time

# ==============================
# SETTINGS
# ==============================

import requests

BOT_TOKEN = "8517200245:AAFUHxeUCCH3GQAuKomqJK-_8YiiHkU0xzk"
CHAT_ID = "1193239390"

def send_telegram(msg):
    try:
        url = f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage"
        requests.post(url, data={
            "chat_id": CHAT_ID,
            "text": msg
        })
    except:
        pass
    
last_heartbeat = time.time()

def send_heartbeat():
    global last_heartbeat
    if time.time() - last_heartbeat > 900:  # 15 minutes
        send_telegram("✅ 1-Min Bot Running Smoothly")
        last_heartbeat = time.time()
        


MODE = "BACKTEST"   # BACKTEST or LIVE
DEBUG = True

SYMBOL = "XAUUSDm"
TIMEFRAME = mt5.TIMEFRAME_M1

LOT_SIZE = 0.02
RR = 3
VOL_LOOKBACK = 10
VOL_MULTIPLIER = 1.5
SWING_LOOKBACK = 5
MAX_SPREAD = 30
MAGICNUMBER=107108
SLBUFFER=2

# Only trade at these exact hours
ALLOWED_HOURS = [1,5,8,13,14,15,16,17,19,21,22] #[1,5,8,12,14,15,16,17,19,21,22,23] #[1,5,8,13,14,15,16,17,19,21,22] #[1,5,8,11,13,14,15,16,17,19,21,22,23]

# ==============================
# MT5 INIT
# ==============================

if not mt5.initialize():
    print("❌ MT5 not connected")
    exit()

# ==============================
# DATA FETCH
# ==============================

def get_data():

    if not mt5.initialize():
        print("❌ MT5 not connected")
        return None

    symbol_info = mt5.symbol_info(SYMBOL)
    if symbol_info is None:
        print(f"❌ Symbol {SYMBOL} not found")
        return None

    if not symbol_info.visible:
        mt5.symbol_select(SYMBOL, True)

    utc_from = datetime.now(pytz.utc) - timedelta(days=90)

    days = 365

    while days >= 30:
        utc_from = datetime.now(pytz.utc) - timedelta(days=days)

        rates = mt5.copy_rates_range(
            SYMBOL,
            TIMEFRAME,
            utc_from,
            datetime.now(pytz.utc)
        )

        if rates is not None and len(rates) > 0:
            print(f"✅ Loaded {days} days of data")
            break

        days -= 30

    if rates is None or len(rates) == 0:
        print("❌ No data available at all")
        return None


    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')

    return df

# ==============================
# STRATEGY LOGIC
# ==============================

def apply_strategy(df):

    df['range'] = df['high'] - df['low']
    df['avg_range'] = df['range'].rolling(VOL_LOOKBACK).mean()
    df['vol_expansion'] = df['range'] > (VOL_MULTIPLIER * df['avg_range'])

    df['prev_low'] = df['low'].rolling(SWING_LOOKBACK).min().shift(1)
    df['prev_high'] = df['high'].rolling(SWING_LOOKBACK).max().shift(1)

    df['sweep_low'] = (df['low'] < df['prev_low']) & (df['close'] > df['prev_low'])
    df['sweep_high'] = (df['high'] > df['prev_high']) & (df['close'] < df['prev_high'])

    df['bos_buy'] = df['close'] > df['high'].shift(1)
    df['bos_sell'] = df['close'] < df['low'].shift(1)

    df['buy'] = df['vol_expansion'] & df['sweep_low'] & df['bos_buy']
    df['sell'] = df['vol_expansion'] & df['sweep_high'] & df['bos_sell']

    return df

def place_market(direction, sl_level):

    tick = mt5.symbol_info_tick(SYMBOL)

    if direction == "BUY":
        price = tick.ask
        sl = sl_level
        tp = price + (price - sl) * RR
        order_type = mt5.ORDER_TYPE_BUY

    else:
        price = tick.bid
        sl = sl_level
        tp = price - (sl - price) * RR
        order_type = mt5.ORDER_TYPE_SELL

    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": SYMBOL,
        "volume": LOT_SIZE,
        "type": order_type,
        "price": price,
        "sl": sl,
        "tp": tp,
        "deviation": 10,
        "magic": MAGICNUMBER,
        "comment": "AlgoScalp",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }

    mt5.order_send(request)
    
def place_pending(direction, entry, sl):

    if direction == "BUY":
        tp = entry + (entry - sl) * RR
        order_type = mt5.ORDER_TYPE_BUY_LIMIT

    else:
        tp = entry - (sl - entry) * RR
        order_type = mt5.ORDER_TYPE_SELL_LIMIT

    request = {
        "action": mt5.TRADE_ACTION_PENDING,
        "symbol": SYMBOL,
        "volume": LOT_SIZE,
        "type": order_type,
        "price": entry,
        "sl": sl,
        "tp": tp,
        "deviation": 10,
        "magic": MAGICNUMBER,
        "comment": "AlgoPending",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_RETURN,
    }

    mt5.order_send(request)
    
def cancel_old_pending():

    orders = mt5.orders_get(symbol=SYMBOL)

    if orders is None:
        return

    now = datetime.now(pytz.utc)

    for order in orders:

        # ✅ ONLY cancel this bot's orders
        if order.magic != MAGICNUMBER:
            continue

        order_time = datetime.fromtimestamp(order.time_setup, pytz.utc)

        if (now - order_time).total_seconds() > 120:
            mt5.order_send({
                "action": mt5.TRADE_ACTION_REMOVE,
                "order": order.ticket
            })
            
def wait_for_new_candle():

    last_time = None

    while True:
        rates = mt5.copy_rates_from_pos(SYMBOL, TIMEFRAME, 0, 2)

        current_time = rates[-1][0]

        if last_time is None:
            last_time = current_time

        if current_time != last_time:
            break

        time.sleep(1)
        
def has_open_position():

    positions = mt5.positions_get(symbol=SYMBOL)

    if positions is None:
        return False

    for pos in positions:
        if pos.magic == MAGICNUMBER:   # ✅ your bot magic
            return True

    return False

def check_entry_and_trade(df, start_idx, entry, sl, tp, trade_type):

    expiry = start_idx + 2   # 2 candles validity

    # Step 1: check if entry gets triggered
    for j in range(start_idx, min(expiry, len(df))):

        high = df.iloc[j]['high']
        low = df.iloc[j]['low']
        time = df.iloc[j]['time']

        if trade_type == "BUY" and low <= entry:
            return simulate_trade(df, j, entry, sl, tp, trade_type)

        if trade_type == "SELL" and high >= entry:
            return simulate_trade(df, j, entry, sl, tp, trade_type)

    # ❌ Not triggered within 2 candles
    return entry, df.iloc[start_idx]['time'], "expired", start_idx

def is_trading_time(current_time):
    return current_time.hour in ALLOWED_HOURS

# ==============================
# BACKTEST ENGINE
# ==============================

def backtest(df):

    trades = []

    i = 20
    trades = []

    while i < len(df) - 1:

        row = df.iloc[i]
        next_row = df.iloc[i+1]
        
        # ⛔ Skip if not allowed hour
        if not is_trading_time(row['time']):
            i += 1
            continue

        if row['buy']:

            move = next_row['open'] - row['low']

            sl = row['low']-SLBUFFER

            # =========================
            # CASE 1: PENDING ORDER
            # =========================
            if move > 10:

                entry = (row['high'] + row['low']) / 2
                tp = entry + (entry - sl) * RR

                exit_price, exit_time, result, exit_index = check_entry_and_trade(
                    df, i+1, entry, sl, tp, "BUY"
                )

                if result == "expired":
                    i += 1
                    continue

            # =========================
            # CASE 2: MARKET ORDER
            # =========================
            else:

                entry = next_row['open']
                tp = entry + (entry - sl) * RR

                exit_price, exit_time, result, exit_index = simulate_trade(
                    df, i+1, entry, sl, tp, "BUY"
                )

            # ✅ STORE TRADE
            trades.append([row['time'], next_row['time'], "BUY", entry, sl, tp,
               exit_time, exit_price, result])

            i = exit_index

        elif row['sell']:

            move = row['high'] - next_row['open']

            sl = row['high']+SLBUFFER

            # =========================
            # CASE 1: PENDING ORDER
            # =========================
            if move > 10:

                entry = (row['high'] + row['low']) / 2
                tp = entry - (sl - entry) * RR

                exit_price, exit_time, result, exit_index = check_entry_and_trade(
                    df, i+1, entry, sl, tp, "SELL"
                )

                if result == "expired":
                    i += 1
                    continue

            # =========================
            # CASE 2: MARKET ORDER
            # =========================
            else:

                entry = next_row['open']
                tp = entry - (sl - entry) * RR

                exit_price, exit_time, result, exit_index = simulate_trade(
                    df, i+1, entry, sl, tp, "SELL"
                )

            # ✅ STORE TRADE
            trades.append([row['time'], next_row['time'], "SELL", entry, sl, tp,
               exit_time, exit_price, result])

            i = exit_index

        else:
            i += 1

    df_trades = pd.DataFrame(trades, columns=[
        "signal_time","entry_time","type","entry","sl","tp",
        "exit_time","exit_price","result"
    ])

    df_trades['pnl'] = np.where(
        df_trades['result']=="win",
        abs(df_trades['tp'] - df_trades['entry']),
        -abs(df_trades['entry'] - df_trades['sl'])
    )

    df_trades.to_excel("backtest_results.xlsx", index=False)

    performance_summary(df_trades)

# ==============================
# TRADE SIMULATION
# ==============================

def simulate_trade(df, start_idx, entry, sl, tp, trade_type):

    for j in range(start_idx, len(df)):

        high = df.iloc[j]['high']
        low = df.iloc[j]['low']
        time = df.iloc[j]['time']

        # =========================
        # BUY TRADE
        # =========================
        if trade_type == "BUY":

            if low <= sl:
                return sl, time, "loss", j

            if high >= tp:
                return tp, time, "win", j

        # =========================
        # SELL TRADE
        # =========================
        elif trade_type == "SELL":

            if high >= sl:
                return sl, time, "loss", j

            if low <= tp:
                return tp, time, "win", j

    return entry, df.iloc[-1]['time'], "no_result", len(df)-1

# ==============================
# PERFORMANCE
# ==============================

def performance_summary(df):

    total = len(df)
    wins = len(df[df['result']=="win"])
    losses = len(df[df['result']=="loss"])

    winrate = wins / total if total > 0 else 0
    accuracy = winrate * 100
    equity = df['pnl'].cumsum()
    drawdown = equity - equity.cummax()
    max_dd = drawdown.min()
    total_pnl = df['pnl'].sum()

    print("\n===== OVERALL PERFORMANCE =====")
    print(f"Total Trades: {total}")
    print(f"Total PnL: {round(total_pnl,2)}")
    print(f"Win Rate: {round(winrate*100,2)}%")
    print(f"Accuracy: {round(accuracy,2)}%")
    print(f"Max Drawdown: {round(max_dd,2)}")

    # Monthly
    df['month'] = df['entry_time'].dt.strftime('%Y-%m')

    monthly = df.groupby('month').agg({
        'pnl': 'sum',
        'result': lambda x: (x == 'win').mean() * 100,
        'entry': 'count'
    }).rename(columns={
        'pnl': 'Total PnL',
        'result': 'Win Rate %',
        'entry': 'Trades'
    })

    print("\n===== MONTHLY PERFORMANCE =====")
    print(monthly)

    # Hourly
    df['hour'] = df['entry_time'].dt.hour

    session = df.groupby('hour').agg({
        'pnl': 'sum',
        'entry': 'count'
    }).rename(columns={
        'pnl': 'Total PnL',
        'entry': 'Trades'
    })

    print("\n===== HOURLY PERFORMANCE =====")
    print(session)

# ==============================
# LIVE LOOP
# ==============================

def run_live():

    print("Running LIVE...")
    send_telegram("🚀 1-Min Scalping Bot Started")

    while True:
        
        send_heartbeat()
        cancel_old_pending()
        wait_for_new_candle()
        
        df = get_data()

        if df is None:
            print("❌ Data fetch failed. Fix MT5 first.")
            mt5.shutdown()
            exit()

        df = apply_strategy(df)

        last = df.iloc[-1]

        tick = mt5.symbol_info_tick(SYMBOL)
        spread = tick.ask - tick.bid
        
        prev = df.iloc[-2]
        current = df.iloc[-1]

        if DEBUG:
            print("\n================ NEW CANDLE ================")
            print(f"Time: {last['time']}")

            print(f"Volatility Expansion: {last['vol_expansion']}")
            print(f"Sweep Low: {last['sweep_low']}")
            print(f"Sweep High: {last['sweep_high']}")
            print(f"BOS Buy: {last['bos_buy']}")
            print(f"BOS Sell: {last['bos_sell']}")
            print(f"Spread: {spread}")

            # ✅ Final conditions
            print(f"BUY Condition: {last['buy']}")
            print(f"SELL Condition: {last['sell']}")
            print(f"Prev Candle BUY Signal: {prev['buy']}")
            print(f"Prev Candle SELL Signal: {prev['sell']}")
            
        if DEBUG:
            if not last['vol_expansion']:
                print("❌ Failed: No Volatility Expansion")

            if not last['sweep_low']:
                print("❌ Failed: No Sweep Low (for BUY)")

            if not last['sweep_high']:
                print("❌ Failed: No Sweep High (for SELL)")

            if not last['bos_buy']:
                print("❌ Failed: No BOS Buy")

            if not last['bos_sell']:
                print("❌ Failed: No BOS Sell")

        if spread > MAX_SPREAD:
            if DEBUG:
                print(f"⛔ Skipped due to HIGH SPREAD: {spread}")
            continue
            
        # ✅ prevent duplicate trades (ONLY this bot)
        if has_open_position():
            if DEBUG:
                print("⛔ Skipped: Already have open position")
            continue

        
        
        # ⛔ Only allow specific hours
        if not is_trading_time(last['time']):
            if DEBUG:
                print(f"⛔ Skipping - not in allowed hours: {last['time'].hour}")
            continue

        if prev['buy']:

            #print("✅ BUY SIGNAL DETECTED")
            msg = f"""
            🚨 BUY SIGNAL
            Time: {last['time']}
            Symbol: {SYMBOL}
            """
            print(msg)
            send_telegram(msg)

            move = current['open'] - prev['low']

            if move > 10:
                msg = "➡️ BUY LIMIT ORDER PLACED"
                print(msg)
                send_telegram(msg)
                entry = (prev['high'] + prev['low']) / 2
                place_pending("BUY", entry, prev['low']-SLBUFFER)

            else:
                msg = "➡️ BUY MARKET ORDER EXECUTED"
                print(msg)
                send_telegram(msg)
                place_market("BUY", prev['low']-SLBUFFER)

        elif prev['sell']:

            #print("✅ SELL SIGNAL DETECTED")
            msg = f"""
            🚨 SELL SIGNAL
            Time: {last['time']}
            Symbol: {SYMBOL}
            """
            print(msg)
            send_telegram(msg)

            move = prev['high'] - current['open']

            if move > 10:
                msg = "➡️ SELL LIMIT ORDER PLACED"
                print(msg)
                send_telegram(msg)
                entry = (prev['high'] + prev['low']) / 2
                place_pending("SELL", entry, prev['high']+SLBUFFER)

            else:
                msg = "➡️ SELL MARKET ORDER EXECUTED"
                print(msg)
                send_telegram(msg)
                place_market("SELL", prev['high']+SLBUFFER)

        #wait_for_new_candle()

# ==============================
# RUN
# ==============================

df = get_data()

if df is None:
    print("❌ Skipping - no data")
    exit()

df = apply_strategy(df)

try:
    if MODE == "BACKTEST":
        backtest(df)

    elif MODE == "LIVE":
        run_live()

except Exception as e:
    send_telegram(f"❌ Bot Crashed: {e}")